In [0]:
CREATE OR REPLACE TABLE retail_lakehouse.gold.gold_daily_sales AS
WITH daily AS (
    SELECT
        invoice_date,

        SUM(
            CASE
                WHEN transaction_type = 'SALE'
                THEN line_amount
                ELSE 0
            END
        ) AS gross_sales,

        SUM(
            CASE
                WHEN transaction_type = 'CANCELLATION'
                THEN ABS(line_amount)
                ELSE 0
            END
        ) AS cancellation_value,

        COUNT(DISTINCT CASE
            WHEN transaction_type = 'SALE'
            THEN invoice_id
        END) AS sale_invoices,

        SUM(
            CASE
                WHEN transaction_type = 'SALE'
                THEN quantity
                ELSE 0
            END
        ) AS units_sold

    FROM retail_lakehouse.retail.silver_retail_transactions
    GROUP BY invoice_date
)

SELECT
    invoice_date,
    gross_sales,
    cancellation_value,
    gross_sales - cancellation_value AS net_sales,
    sale_invoices,
    units_sold,
    gross_sales / NULLIF(sale_invoices, 0) AS average_order_value
FROM daily;

SELECT *
FROM retail_lakehouse.gold.gold_daily_sales
ORDER BY invoice_date;

SELECT
    SUM(gross_sales) AS gross_sales,
    SUM(cancellation_value) AS cancellation_value,
    SUM(net_sales) AS net_sales,
    SUM(sale_invoices) AS daily_invoice_counts,
    SUM(units_sold) AS units_sold
FROM retail_lakehouse.gold.gold_daily_sales;